#Índices y brecha territorial

Equipo UBiK2 · Data Jam CVP 2026

**Objetivo:** unir la presencia de la CVP (notebook 01), la necesidad de la Encuesta Multipropósito 2021 (notebook 02) y el riesgo por movimientos en masa (notebook 01) en una sola tabla de 95 dominios, normalizar las variables, calcular el índice de necesidad, el índice de presencia y la brecha, y generar el archivo final para el visor.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q geopandas
import os
import geopandas as gpd, pandas as pd

BASE = "/content/drive/MyDrive/UBiK2_DataJam"
PROC = f"{BASE}/datos_procesados"
GPKG = f"{PROC}/ubik2.gpkg"
WEB  = f"{PROC}/web"

Mounted at /content/drive


##1. Tabla maestra de 95 dominios

Se unen tres productos por cod_upz_grupo:

upz_presencia_95 (GeoPackage): geometría, hogares, conteos y tasas de presencia de las cinco líneas.
riesgo_95.csv: porcentaje del área de cada dominio en amenaza alta, media y baja.
indicadores_necesidad_95_dominios.xlsx, hoja Detalle: proporción de hogares en cada condición de necesidad y sus componentes.

In [2]:
presencia = gpd.read_file(GPKG, layer="upz_presencia_95")
presencia = presencia.drop(columns=[c for c in presencia.columns if c.startswith("pct_amenaza")])
riesgo = pd.read_csv(f"{PROC}/riesgo_95.csv")
nec = pd.read_excel(f"{PROC}/indicadores_necesidad_95_dominios.xlsx", sheet_name="Detalle")

# De la necesidad solo se toman la llave y los indicadores (hogares y nombres ya están en presencia)
nec = nec[["COD_UPZ_GRUPO"] + [c for c in nec.columns if c.startswith("IND_")]]
nec.columns = [c.lower() for c in nec.columns]

upz = (presencia
       .merge(riesgo, on="cod_upz_grupo", how="left", validate="one_to_one")
       .merge(nec, on="cod_upz_grupo", how="left", validate="one_to_one"))

NECESIDAD = ["ind_tenencia", "ind_vivienda", "ind_entorno", "pct_amenaza_alta", "ind_vulnerabilidad"]
PRESENCIA = ["tit_x1000hog", "viv_x1000hog", "cps_x1000hog", "barrios_x1000hog", "reas_x1000hog"]

assert len(upz) == 95
print("Columnas:", upz.shape[1])
print("Vacíos en las variables del índice:", upz[NECESIDAD + PRESENCIA].isna().sum().sum())
upz[NECESIDAD + PRESENCIA].describe().round(3).T

Columnas: 41
Vacíos en las variables del índice: 0


,count,mean,std,min,25%,50%,75%,max
ind_tenencia,95.0,0.091,0.045,0.023,0.062,0.079,0.106,0.258
ind_vivienda,95.0,0.089,0.069,0.004,0.034,0.074,0.130,0.338
ind_entorno,95.0,0.169,0.141,0.024,0.073,0.133,0.206,0.835
pct_amenaza_alta,95.0,3.677,6.732,0.000,0.000,0.000,3.440,30.450
ind_vulnerabilidad,95.0,0.152,0.088,0.032,0.077,0.131,0.222,0.355
tit_x1000hog,95.0,2.187,10.579,0.000,0.000,0.000,0.085,91.580
viv_x1000hog,95.0,0.160,0.706,0.000,0.000,0.000,0.000,5.770
cps_x1000hog,95.0,0.522,2.060,0.000,0.000,0.000,0.000,14.640
barrios_x1000hog,95.0,0.082,0.310,0.000,0.000,0.000,0.000,2.370
reas_x1000hog,95.0,2.917,9.627,0.000,0.000,0.000,0.480,66.000


##2. Normalización

Las diez variables se llevan a una escala de 0 a 1 para poder combinarlas. Las tasas de presencia están muy concentradas: en las cinco líneas la mediana es 0, y unas pocas UPZ tienen valores muy altos (Titulación en Lucero llega a 91,6 por 1.000 hogares, cuando el 75 % de las UPZ está por debajo de 0,1). Con un min-max simple, ese valor extremo dejaría a casi todas las UPZ cerca de 0.

Por eso, antes de normalizar, cada variable se limita a su percentil 95 (winsorización, como recomienda la guía de la OCDE para indicadores compuestos). Los valores por encima se tratan como el máximo. Luego se aplica min-max entre el mínimo y ese tope. Se usa el mismo método en las diez variables.

In [3]:
def normalizar(s, p=0.95):
    """Min-max de 0 a 1 con tope en el percentil p."""
    tope = s.quantile(p)
    if tope == s.min():          # si el percentil coincide con el mínimo, se usa el máximo
        tope = s.max()
    return ((s.clip(upper=tope) - s.min()) / (tope - s.min())).round(4)

for col in NECESIDAD + PRESENCIA:
    upz[f"{col}_n"] = normalizar(upz[col])

# Comparación con min-max simple, para ver el efecto del tope
simple = (upz["tit_x1000hog"] - upz["tit_x1000hog"].min()) / (upz["tit_x1000hog"].max() - upz["tit_x1000hog"].min())
print("Titulación · UPZ con valor normalizado > 0,2")
print("  min-max simple:", (simple > 0.2).sum(), "| con tope p95:", (upz["tit_x1000hog_n"] > 0.2).sum(), "\n")

upz[[f"{c}_n" for c in NECESIDAD + PRESENCIA]].describe().round(3).T

Titulación · UPZ con valor normalizado > 0,2
  min-max simple: 4 | con tope p95: 12 



,count,mean,std,min,25%,50%,75%,max
ind_tenencia_n,95.0,0.398,0.247,0.0,0.234,0.334,0.499,1.0
ind_vivienda_n,95.0,0.387,0.282,0.0,0.140,0.331,0.599,1.0
ind_entorno_n,95.0,0.372,0.269,0.0,0.138,0.308,0.518,1.0
pct_amenaza_alta_n,95.0,0.186,0.327,0.0,0.000,0.000,0.186,1.0
ind_vulnerabilidad_n,95.0,0.441,0.317,0.0,0.167,0.369,0.710,1.0
tit_x1000hog_n,95.0,0.101,0.252,0.0,0.000,0.000,0.016,1.0
viv_x1000hog_n,95.0,0.085,0.252,0.0,0.000,0.000,0.000,1.0
cps_x1000hog_n,95.0,0.115,0.288,0.0,0.000,0.000,0.000,1.0
barrios_x1000hog_n,95.0,0.102,0.263,0.0,0.000,0.000,0.000,1.0
reas_x1000hog_n,95.0,0.109,0.268,0.0,0.000,0.000,0.031,1.0


##3. Índices y brecha
**Índice de necesidad:** promedio de las cinco dimensiones normalizadas (tenencia, vivienda, entorno, riesgo y vulnerabilidad), con pesos iguales.

**Índice de presencia:** promedio de las cinco líneas de la CVP normalizadas, con pesos iguales.

**Brecha:** necesidad menos presencia. Va de −1 a 1; un valor alto indica mucha necesidad y poca presencia.

**Brecha por línea:** cada dimensión de necesidad frente a la línea que la atiende: tenencia–Titulación, vivienda–Mejoramiento de Vivienda y Curaduría Pública Social, entorno–Mejoramiento de Barrios, riesgo–Reasentamientos. Para vivienda se suman las tasas de las dos líneas, que se miden igual (predios por 1.000 hogares), y se normaliza la suma. La vulnerabilidad no tiene línea propia: entra al índice de necesidad como dimensión transversal.

**Prioridad:** posición de cada UPZ según la brecha (1 = la mayor) y una categoría por quintiles para la leyenda del visor.

In [4]:
N = [f"{c}_n" for c in NECESIDAD]
P = [f"{c}_n" for c in PRESENCIA]

# Índices y brecha general
upz["indice_necesidad"] = upz[N].mean(axis=1).round(4)
upz["indice_presencia"] = upz[P].mean(axis=1).round(4)
upz["brecha"] = (upz["indice_necesidad"] - upz["indice_presencia"]).round(4)

# Brecha por línea
upz["viv_cps_x1000hog_n"] = normalizar(upz["viv_x1000hog"] + upz["cps_x1000hog"])
upz["brecha_tenencia"] = (upz["ind_tenencia_n"] - upz["tit_x1000hog_n"]).round(4)
upz["brecha_vivienda"] = (upz["ind_vivienda_n"] - upz["viv_cps_x1000hog_n"]).round(4)
upz["brecha_entorno"]  = (upz["ind_entorno_n"] - upz["barrios_x1000hog_n"]).round(4)
upz["brecha_riesgo"]   = (upz["pct_amenaza_alta_n"] - upz["reas_x1000hog_n"]).round(4)

# Prioridad y categoría
upz["prioridad"] = upz["brecha"].rank(ascending=False, method="min").astype(int)
upz["categoria_brecha"] = pd.qcut(upz["brecha"], 5,
                                  labels=["Muy baja", "Baja", "Media", "Alta", "Muy alta"]).astype(str)

# ¿La CVP está donde está la necesidad?
rho = upz["indice_necesidad"].corr(upz["indice_presencia"], method="spearman")
print(f"Correlación de Spearman necesidad–presencia: {rho:.2f}\n")

cols = ["prioridad", "cod_upz_grupo", "NOMBRE_UPZ_GRUPO", "indice_necesidad", "indice_presencia",
        "brecha", "integralidad", "brecha_tenencia", "brecha_vivienda", "brecha_entorno", "brecha_riesgo"]
upz.sort_values("prioridad")[cols].head(15)

Correlación de Spearman necesidad–presencia: 0.75



,prioridad,cod_upz_grupo,NOMBRE_UPZ_GRUPO,indice_necesidad,indice_presencia,brecha,integralidad,brecha_tenencia,brecha_vivienda,brecha_entorno,brecha_riesgo
65,1,89,San Isidro - Patios,0.9420,0.0383,0.9037,1,1.0000,1.0000,1.0000,0.8084
59,2,82,Patio Bonito,0.5676,0.0242,0.5434,2,0.2792,0.8227,0.7154,-0.1003
1,3,11,San Cristóbal Norte,0.6181,0.0792,0.5389,2,0.8576,0.5727,0.1052,0.5710
38,4,54,Marruecos,0.6632,0.1514,0.5118,4,0.3992,0.4880,0.4915,0.7868
37,5,53,Marco Fidel Suárez,0.5667,0.0642,0.5025,3,0.2717,0.5267,0.6930,0.0622
92,6,815,CIUDAD BOLÍVAR: Monteblanco + El Tesoro + El M...,0.9375,0.4411,0.4964,3,0.0790,0.5200,1.0000,0.0000
42,7,62,Tunjuelito,0.4903,0.0000,0.4903,0,0.6181,0.8404,0.0148,0.0595
64,8,87,Tintál Sur,0.4718,0.0000,0.4718,0,1.0000,0.3293,0.2871,0.0000
61,9,84,Bosa Occidental,0.4857,0.0142,0.4715,2,0.2359,0.6929,0.6035,-0.0013
56,10,79,Calandaima,0.4595,0.0000,0.4595,0,0.9913,0.1794,0.3988,0.0000


In [5]:
# Cuadrantes respecto a la mediana
med_n, med_p = upz["indice_necesidad"].median(), upz["indice_presencia"].median()
alta_n, alta_p = upz["indice_necesidad"] >= med_n, upz["indice_presencia"] > med_p
upz["cuadrante"] = "Baja necesidad, baja presencia"
upz.loc[alta_n & ~alta_p, "cuadrante"] = "Alta necesidad, baja presencia"
upz.loc[alta_n & alta_p, "cuadrante"] = "Alta necesidad, alta presencia"
upz.loc[~alta_n & alta_p, "cuadrante"] = "Baja necesidad, alta presencia"
print(upz["cuadrante"].value_counts(), "\n")

# Exportación del producto final
upz.to_file(GPKG, layer="upz_brecha_95", driver="GPKG")
upz.drop(columns="geometry").to_csv(f"{PROC}/upz_brecha_95.csv", index=False, encoding="utf-8-sig")

def exportar_web(gdf, nombre, tolerancia_m=3):
    capa = gdf.copy()
    capa["geometry"] = capa.geometry.simplify(tolerancia_m, preserve_topology=True)
    capa = capa.to_crs(epsg=4326)
    ruta = f"{WEB}/{nombre}.geojson"
    capa.to_file(ruta, driver="GeoJSON", COORDINATE_PRECISION=6)
    print(f"{nombre}.geojson · {len(capa)} elementos · {os.path.getsize(ruta)/1024:.0f} KB")

exportar_web(upz, "upz_brecha_95")
print(upz.shape)

cuadrante
Baja necesidad, baja presencia    38
Alta necesidad, alta presencia    37
Alta necesidad, baja presencia    11
Baja necesidad, alta presencia     9
Name: count, dtype: int64 

upz_brecha_95.geojson · 95 elementos · 410 KB
(95, 62)


**Resultados:** La correlación de Spearman entre el índice de necesidad y el de presencia es 0,75: en general, la CVP interviene más donde hay más necesidad. La brecha identifica las excepciones. San Isidro-Patios (cerros orientales) tiene la brecha más alta de la ciudad (0,90), con la mayor necesidad de los 95 dominios y presencia casi nula en las cinco líneas. La brecha alta tiene dos perfiles: UPZ de ladera con necesidad severa y presencia parcial (San Isidro-Patios, San Cristóbal Norte, Marruecos, Monteblanco–El Tesoro–El Mochuelo, Lourdes), y UPZ planas del sur y el occidente con necesidad media y sin presencia en las capas abiertas (Patio Bonito, Tunjuelito, Tintal Sur, Bosa Occidental). En estas últimas, el riesgo por inundación no está medido.

In [6]:
upz["cuadrante"] = upz["cuadrante"].str.replace("alta presencia", "con presencia").str.replace("baja presencia", "sin presencia")
print(upz["cuadrante"].value_counts())
print(upz.loc[upz["cuadrante"] == "Alta necesidad, sin presencia", "NOMBRE_UPZ_GRUPO"].tolist())

cuadrante
Baja necesidad, sin presencia    38
Alta necesidad, con presencia    37
Alta necesidad, sin presencia    11
Baja necesidad, con presencia     9
Name: count, dtype: int64
['Santa Isabel', 'Quiroga', 'Tunjuelito', 'Engativá', 'Fontibón San Pablo', 'Zona Franca', 'Calandaima', 'Gran Britalia', 'Las Margaritas', 'Tintál Sur', 'La Sabana']


In [7]:
alertas = upz[upz["cuadrante"] == "Alta necesidad, sin presencia"].sort_values("indice_necesidad", ascending=False)
alertas[["NOMBRE_UPZ_GRUPO", "localidad", "indice_necesidad", "ind_tenencia", "ind_vivienda",
         "ind_entorno", "pct_amenaza_alta", "ind_vulnerabilidad"]].round(3)

,NOMBRE_UPZ_GRUPO,localidad,indice_necesidad,ind_tenencia,ind_vivienda,ind_entorno,pct_amenaza_alta,ind_vulnerabilidad
42,Tunjuelito,Tunjuelito,0.490,0.126,0.180,0.029,1.1,0.278
64,Tintál Sur,Bosa,0.472,0.214,0.073,0.125,0.0,0.231
56,Calandaima,Kennedy,0.460,0.187,0.042,0.165,0.0,0.227
58,Gran Britalia,Kennedy,0.440,0.049,0.129,0.290,0.0,0.218
60,Las Margaritas,Kennedy,0.434,0.101,0.044,0.292,0.0,0.233
53,Fontibón San Pablo,Fontibón,0.407,0.054,0.127,0.261,0.0,0.191
24,Quiroga,Rafael Uribe Uribe,0.396,0.097,0.111,0.143,0.0,0.215
22,Santa Isabel,Los Mártires,0.372,0.094,0.130,0.102,0.0,0.195
54,Zona Franca,Fontibón,0.352,0.099,0.059,0.266,0.0,0.127
51,Engativá,Engativá,0.348,0.062,0.119,0.156,0.0,0.188


**Alertas territoriales:** Once dominios tienen necesidad por encima de la mediana y ninguna presencia en las capas abiertas de la CVP: Santa Isabel, Quiroga, Tunjuelito, Engativá, Fontibón San Pablo, Zona Franca, Calandaima, Gran Britalia, Las Margaritas, Tintal Sur y La Sabana. Todos están en la ciudad plana y consolidada, fuera de la periferia de ladera donde se concentra la acción de la entidad. Se presentan como alertas para revisar, no como prioridades definitivas: su necesidad es moderada; en algunos casos, como los inquilinatos del centro, corresponde a políticas de otras entidades; y las capas abiertas no incluyen todas las intervenciones de la CVP ni el riesgo por inundación.